# 🎨 Data Designer Tutorial: The Basics

#### 📚 What you'll learn

This notebook demonstrates the basics of Data Designer by generating a "Fraud Detection Conversation" dataset.

Example is alike:

```
Customer:
Why was my credit card declined?

Agent:
It looks like there were multiple overseas transactions.

Customer:
I didn't make those purchases.

Agent:
We'll block the card immediately.
```

Data Designer can generate:
```
Different ages

Different countries

Different amounts

Different merchants

Different languages

Different scam methods
```
for example:
```
ATM Cash Withdrawal

POS Purchase

Apple Pay

Google Pay

Wire Transfer

Crypto Exchange

Gift Card Scam

Romance Scam

Investment Scam

SIM Swap
```

This ultimately yields tens of thousands of Fraud Conversations.

This can be directly trained for:

```
LLM, Fraud Classifier, Call Summary, and Risk Detection.
```


このノートブックでは、「不正検出会話」データセットを生成することで、Data Designerの基本操作を説明します。

例：

```
顧客：
なぜクレジットカードが拒否されたのですか？

担当者：
海外での取引が複数件発生しているようです。

顧客：
私はそれらの購入をしていません。

担当者：
すぐにカードを停止します。

```

Data Designerでは、以下のようなデータを生成できます。
```
年齢層

国籍

金額

加盟店

言語

詐欺の手口
```

（例：
```
ATMでの現金引き出し

POS端末での購入

Apple Pay

Google Pay

銀行振込

仮想通貨取引所

ギフトカード詐欺

ロマンス詐欺

投資詐欺

SIMスワップ詐欺
```

最終的に、数万件もの不正検出会話データが生成されます。


これは以下の用途に直接学習させることができます。

```
LLM、不正分類器、通話概要、リスク検出

```


### 📦 Import Data Designer

- `data_designer.config` provides access to the configuration API.

- `DataDesigner` is the main interface for data generation.


- `data_designer.config` は設定APIへのアクセスを提供します。

- `DataDesigner` はデータ生成のための主要なインターフェースです。

In [1]:
# ! export NVIDIA_API_KEY="" # TODO copy your nvidia-api-key here:
# https://build.nvidia.com/settings/api-keys
# login with you email box (or register)
# click "Generate API key"
import os

os.environ["NVIDIA_API_KEY"] = "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" # TODO replace with your true api key!!!
print(os.getenv("NVIDIA_API_KEY"))


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


In [2]:
# ! pip install data-designer

In [3]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

ModuleNotFoundError: No module named 'data_designer'

### ⚙️ Initialize the Data Designer interface

- `DataDesigner` is the main object responsible for managing the data generation process.

- When initialized without arguments, the [default model providers](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) are used.


### ⚙️ データデザイナーインターフェースの初期化

- `DataDesigner` は、データ生成プロセスを管理する主要なオブジェクトです。

- 引数を指定せずに初期化した場合、[デフォルトのモデルプロバイダー](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) が使用されます。

In [4]:
data_designer = DataDesigner()

### 🎛️ Define model configurations

- Each `ModelConfig` defines a model that can be used during the generation process.

- The "model alias" is used to reference the model in the Data Designer config (as we will see below).

- The "model provider" is the external service that hosts the model (see the [model config](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) docs for more details).

- By default, we use [build.nvidia.com](https://build.nvidia.com/models) as the model provider.


### 🎛️ モデル設定の定義

- 各 `ModelConfig` は、生成プロセスで使用できるモデルを定義します。

- 「モデルエイリアス」は、Data Designer の設定でモデルを参照するために使用されます（後述します）。

- 「モデルプロバイダ」は、モデルをホストする外部サービスです（詳細は [モデル設定](https://nvidia-nemo.github.io/DataDesigner/latest/concepts/models/default-model-settings/) のドキュメントを参照してください）。

- デフォルトでは、モデルプロバイダとして [build.nvidia.com](https://build.nvidia.com/models) を使用します。


In [5]:
# This name is set in the model provider configuration.
MODEL_PROVIDER = "nvidia"

# The model ID is from build.nvidia.com.
MODEL_ID = "nvidia/nemotron-3-nano-30b-a3b"

# We choose this alias to be descriptive for our use case.
MODEL_ALIAS = "nemotron-nano-v3"

model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_ID,
        provider=MODEL_PROVIDER,
        inference_parameters=dd.ChatCompletionInferenceParams(
            temperature=1.0,
            top_p=1.0,
            max_tokens=2048,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        ),
    )
]

### 🏗️ Initialize the Data Designer Config Builder

- The Data Designer config defines the dataset schema and generation process.

- The config builder provides an intuitive interface for building this configuration.

- The list of model configs is provided to the builder at initialization.

### 🏗️ データ デザイナー構成ビルダーを初期化する

- データ デザイナー設定は、データセット スキーマと生成プロセスを定義します。

- 構成ビルダーは、この構成を構築するための直感的なインターフェイスを提供します。

- モデル構成のリストは、初期化時にビルダーに提供されます。


In [6]:
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

## 🎲 Getting started with sampler columns

- Sampler columns offer non-LLM based generation of synthetic data.

- They are particularly useful for **steering the diversity** of the generated data, as we demonstrate below.

<br>

You can view available samplers using the config builder's `info` property:


## 🎲 サンプラー列の入門

- サンプラー列は、LLMに基づかない合成データの生成を可能にします。

- 以下で示すように、サンプラー列は生成されるデータの**多様性**を**制御するのに特に役立ちます。

<br>

利用可能なサンプラーは、設定ビルダーの`info`プロパティで確認できます。

In [7]:
config_builder.info.display("samplers")

─────────────────────────────────────────── NeMo Data Designer Samplers ───────────────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Type               ┃ Parameter                ┃ Data Type                         ┃ Required ┃ Constraints      ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ bernoulli          │ p                        │ number                            │    ✓     │ >= 0.0, <= 1.0   │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ bernoulli_mixture  │ p                        │ number                            │    ✓     │ >= 0.0, <= 1.0   │
│                    │ dist_name                │ string                            │    ✓     │                  │
│                    │ dist_params              │ dict                              │    ✓     │                  │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ binomial           │ n                        │ integer                           │    ✓     │                  │
│                    │ p                        │ number                            │    ✓     │ >= 0.0, <= 1.0   │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ category           │ values                   │ string[] | integer[] | number[]   │    ✓     │ len > 1          │
│                    │ weights                  │ number[] | null                   │          │                  │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ datetime           │ start                    │ string                            │    ✓     │                  │
│                    │ end                      │ string                            │    ✓     │                  │
│                    │ unit                     │ string                            │          │                  │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ gaussian           │ mean                     │ number                            │    ✓     │                  │
│                    │ stddev                   │ number                            │    ✓     │                  │
│                    │ decimal_places           │ integer | null                    │          │                  │
│                    │ sampler_type             │ string                            │          │                  │
├────────────────────┼──────────────────────────┼───────────────────────────────────┼──────────┼──────────────────┤
│ person             │ locale                   │ string                            │          │                  │
│                    │ sex                      │ string | null                     │          │                  │
│                    │ city                     │ string | string[] | null          │          │                  │
│                    │ age_range                │ integer[]                         │          │ len > 2, len < 2 │
│                    │ select_field_values      │ objec

Let's start designing our product review dataset by adding product category and subcategory columns.

製品レビューデータセットの設計を始めるにあたり、まずは製品カテゴリとサブカテゴリの列を追加しましょう。


In [8]:
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="product_category",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "Electronics",
                "Clothing",
                "Home & Kitchen",
                "Books",
                "Home Office",
            ],
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="product_subcategory",
        sampler_type=dd.SamplerType.SUBCATEGORY,
        params=dd.SubcategorySamplerParams(
            category="product_category",
            values={
                "Electronics": [
                    "Smartphones",
                    "Laptops",
                    "Headphones",
                    "Cameras",
                    "Accessories",
                ],
                "Clothing": [
                    "Men's Clothing",
                    "Women's Clothing",
                    "Winter Coats",
                    "Activewear",
                    "Accessories",
                ],
                "Home & Kitchen": [
                    "Appliances",
                    "Cookware",
                    "Furniture",
                    "Decor",
                    "Organization",
                ],
                "Books": [
                    "Fiction",
                    "Non-Fiction",
                    "Self-Help",
                    "Textbooks",
                    "Classics",
                ],
                "Home Office": [
                    "Desks",
                    "Chairs",
                    "Storage",
                    "Office Supplies",
                    "Lighting",
                ],
            },
        ),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="target_age_range",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(values=["18-25", "25-35", "35-50", "50-65", "65+"]),
    )
)

# Optionally validate that the columns are configured correctly.
data_designer.validate(config_builder)

[06:13:33] [INFO] ✅ Validation passed


Next, let's add samplers to generate data related to the customer and their review.


次に、顧客とそのレビューに関連するデータを生成するためのサンプラーを追加しましょう。


In [9]:
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="customer",
        sampler_type=dd.SamplerType.PERSON_FROM_FAKER,
        params=dd.PersonFromFakerSamplerParams(age_range=[18, 70], locale="en_US"),
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="number_of_stars",
        sampler_type=dd.SamplerType.UNIFORM,
        params=dd.UniformSamplerParams(low=1, high=5),
        convert_to="int",  # Convert the sampled float to an integer.
    )
)

config_builder.add_column(
    dd.SamplerColumnConfig(
        name="review_style",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["rambling", "brief", "detailed", "structured with bullet points"],
            weights=[1, 2, 2, 1],
        ),
    )
)

data_designer.validate(config_builder)

[06:13:33] [INFO] ✅ Validation passed


## 🦜 LLM-generated columns

- The real power of Data Designer comes from leveraging LLMs to generate text, code, and structured data.

- When prompting the LLM, we can use Jinja templating to reference other columns in the dataset.

- As we see below, nested json fields can be accessed using dot notation.


## 🦜 LLMで生成された列

- Data Designerの真価は、LLMを活用してテキスト、コード、構造化データを生成することにあります。

- LLMを実行する際、Jinjaテンプレートを使用してデータセット内の他の列を参照できます。

- 以下に示すように、ネストされたJSONフィールドにはドット表記でアクセスできます。

In [10]:
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="product_name",
        prompt=(
            "You are a helpful assistant that generates product names. DO NOT add quotes around the product name.\n\n"
            "Come up with a creative product name for a product in the '{{ product_category }}' category, focusing "
            "on products related to '{{ product_subcategory }}'. The target age range of the ideal customer is "
            "{{ target_age_range }} years old. Respond with only the product name, no other text."
        ),
        model_alias=MODEL_ALIAS,
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="customer_review",
        prompt=(
            "You are a customer named {{ customer.first_name }} from {{ customer.city }}, {{ customer.state }}. "
            "You are {{ customer.age }} years old and recently purchased a product called {{ product_name }}. "
            "Write a review of this product, which you gave a rating of {{ number_of_stars }} stars. "
            "The style of the review should be '{{ review_style }}'. "
            "Respond with only the review, no other text."
        ),
        model_alias=MODEL_ALIAS,
    )
)

data_designer.validate(config_builder)

[06:13:33] [INFO] ✅ Validation passed


### 🔁 Iteration is key – preview the dataset!

1. Use the `preview` method to generate a sample of records quickly.

2. Inspect the results for quality and format issues.

3. Adjust column configurations, prompts, or parameters as needed.

4. Re-run the preview until satisfied.

### 🔁 繰り返し検証が鍵です – データセットをプレビューしましょう！

1. `preview` メソッドを使用して、レコードのサンプルをすばやく生成します。

2. 結果の品質とフォーマットに問題がないか確認します。

3. 必要に応じて、列の設定、プロンプト、またはパラメータを調整します。

4. 満足できるまでプレビューを繰り返し実行します。


In [11]:
preview = data_designer.preview(config_builder, num_records=2)

[06:13:33] [INFO] 🔭 Preview generation in progress
[06:13:33] [INFO]   |-- 🔒 Jinja rendering engine: secure
[06:13:33] [INFO] ✅ Validation passed
[06:13:33] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[06:13:33] [INFO] 🩺 Running health checks for models...
[06:13:33] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[06:13:33] [INFO]   |-- ✅ Passed!
[06:13:33] [INFO] ⚡ DATA_DESIGNER_ASYNC_ENGINE is enabled - using async task-queue preview
[06:13:33] [INFO] 📝 llm-text model config for column 'product_name'
[06:13:33] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[06:13:33] [INFO]   |-- model alias: 'nemotron-nano-v3'
[06:13:33] [INFO]   |-- model provider: 'nvidia'
[06:13:33] [INFO]   |-- inference parameters:
[06:13:33] [INFO]   |  |-- generation_type=chat-completion
[06:13:33] [INFO]   |  |-- max_parallel_requests=4
[06:13:33] [INFO]   |  |-- extra_body={'chat_template_kwargs': {'enable_t

In [12]:
# Run this cell multiple times to cycle through the 2 preview records.
preview.display_sample_record()

                                              Generated Columns                                               
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                ┃ Value                                                                                ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ product_category    │ Home Office                                                                          │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ product_subcategory │ Desks                                                                                │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ target_age_range    │ 35-50                                                                                │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ customer            │ {                                                                                    │
│                     │     'uuid': '3b1ad8a3-5203-4eb9-af23-f84f3a32c39a',                                  │
│                     │     'locale': 'en_US',                                                               │
│                     │     'first_name': 'Richard',                                                         │
│                     │     'last_name': 'Kelly',                                                            │
│                     │     'middle_name': None,                                                             │
│                     │     'sex': 'Male',                                                                   │
│                     │     'street_number': '9351',                                                         │
│                     │     'street_name': 'Taylor Falls',                                                   │
│                     │     'city': 'Petersonport',                                                          │
│                     │     'state': 'Florida',                                                              │
│                     │     'postcode': '11303',                                                             │
│                     │     'age': 68,                                                                       │
│                     │     'birth_date': '1958-06-04',                                                      │
│                     │     'country': "Lao People's Democratic Republic",                                   │
│                     │     'marital_status': 'widowed',                                                     │
│                     │     'education_level': 'secondary_education',                                        │
│                     │     'unit': '',                                                                      │
│                     │     'occupation': 'Colour technologist',                                             │
│                     │     'phone_number': '+1-287-493-7360x305',                                           │
│                     │     'bachelors_field': 'no_degree'                                                   │
│                     │ }                                                                                    │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ number_of_stars     │ 3                                                                                    │
├─────────────────────┼──────────────────────────────────────────────────────────────────────────────────────┤
│ review_style        │ brief                                                                                │
├───

In [13]:
# The preview dataset is available as a pandas DataFrame.
preview.dataset

,product_category,product_subcategory,target_age_range,customer,number_of_stars,review_style,product_name,customer_review
0,Home Office,Desks,35-50,{'uuid': '3b1ad8a3-5203-4eb9-af23-f84f3a32c39a...,3,brief,Strategic Workspace Desk,I purchased the Strategic Workspace Desk as a ...
1,Clothing,Accessories,65+,{'uuid': '6c40591b-3cec-4554-8840-c5dc9b8150ff...,1,detailed,Timeless Threads Knit‑Fit Scarf,"I am Samantha, 58 years old, from Monroeburgh,..."


### 📊 Analyze the generated data

- Data Designer automatically generates a basic statistical analysis of the generated data.

- This analysis is available via the `analysis` property of generation result objects.

### 📊 生成データの分析

- データデザイナーは、生成データの基本的な統計分析を自動的に生成します。

- この分析結果は、生成結果オブジェクトの `analysis` プロパティから利用できます。


In [14]:
# Print the analysis as a table.
preview.analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 2                               │ 8                               │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                    ┃       data type ┃            number unique values ┃               sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ product_category               │          string │                      2 (100.0%) │                   category │
├────────────────────────────────┼─────────────────┼─────────────────────────────────┼────────────────────────────┤
│ product_subcategory            │          string │                      2 (100.0%) │                subcategory │
├────────────────────────────────┼─────────────────┼─────────────────────────────────┼────────────────────────────┤
│ target_age_range               │          string │                      2 (100.0%) │                   category │
├────────────────────────────────┼─────────────────┼─────────────────────────────────┼────────────────────────────┤
│ customer                       │            dict │                      2 (100.0%) │          person_from_faker │
├────────────────────────────────┼─────────────────┼─────────────────────────────────┼────────────────────────────┤
│ number_of_stars                │             int │                      2 (100.0%) │                    uniform │
├────────────────────────────────┼─────────────────┼─────────────────────────────────┼────────────────────────────┤
│ review_style                   │          string │                      2 (100.0%) │                   category │
└────────────────────────────────┴─────────────────┴─────────────────────────────────┴────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Text Columns                                                
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                       ┃               ┃                            ┃     prompt tokens ┃      completion tokens ┃
┃ column name           ┃     data type ┃       number unique values ┃        per record ┃             per record ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ product_name          │        string │                 2 (100.0%) │      73.5 +/- 0.5 │            6.5 +/- 3.5 │
├───────────────────────┼───────────────┼─────────────────

### 🆙 Scale up!

- Happy with your preview data?

- Use the `create` method to submit larger Data Designer generation jobs.

### 🆙 スケールアップ！

- プレビューデータに満足いただけましたか？

- `create` メソッドを使用して、より大規模な Data Designer 生成ジョブを送信してください。


In [15]:
results = data_designer.create(config_builder, num_records=10, dataset_name="tutorial-1")

[06:13:36] [INFO] 🎨 Creating Data Designer dataset
[06:13:36] [INFO]   |-- 🔒 Jinja rendering engine: secure
[06:13:36] [INFO] 📂 Dataset path '/workspace/asr/brev.nemo.curator.20260324/data_designer/artifacts/tutorial-1' already exists. Dataset from this session
		     will be saved to '/workspace/asr/brev.nemo.curator.20260324/data_designer/artifacts/tutorial-1_06-10-2026_061336' instead.
[06:13:36] [INFO] ✅ Validation passed
[06:13:36] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[06:13:36] [INFO] 🩺 Running health checks for models...
[06:13:36] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[06:13:36] [INFO]   |-- ✅ Passed!
[06:13:36] [INFO] ⚡ DATA_DESIGNER_ASYNC_ENGINE is enabled - using async task-queue builder
[06:13:36] [INFO] 📝 llm-text model config for column 'product_name'
[06:13:36] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[06:13:36] [INFO]   |-- model alias: 'nemotron-nan

In [16]:
# Load the generated dataset as a pandas DataFrame.
dataset = results.load_dataset()

dataset.head()

,product_category,product_subcategory,target_age_range,customer,number_of_stars,review_style,product_name,customer_review
0,Home Office,Chairs,25-35,"{'age': 70, 'bachelors_field': 'no_degree', 'b...",3,rambling,ZenithMesh Office Chair,"Well now, I reckon I ought to put down a few t..."
1,Electronics,Accessories,18-25,"{'age': 24, 'bachelors_field': 'education', 'b...",4,brief,LumiCharge Pulse Grip,"⭐️⭐️⭐️⭐️☆ I’m Teresa, 24, from Baileyborough..."
2,Home Office,Chairs,25-35,"{'age': 57, 'bachelors_field': 'stem', 'birth_...",3,structured with bullet points,ErgoFocus Chair,- Purchased the ErgoFocus Chair as a customer ...
3,Home & Kitchen,Decor,65+,"{'age': 68, 'bachelors_field': 'no_degree', 'b...",3,brief,Crystal Bloom Chime,It looks beautiful but the chime is barely aud...
4,Books,Textbooks,35-50,"{'age': 21, 'bachelors_field': 'no_degree', 'b...",2,detailed,Modern Guide Series: Comprehensive Curriculum ...,I bought the Modern Guide Series: Comprehensiv...


In [17]:
# Load the analysis results into memory.
analysis = results.load_analysis()

analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 10                              │ 8                               │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                    ┃       data type ┃            number unique values ┃               sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ product_category               │          string │                       5 (50.0%) │                   category │
├────────────────────────────────┼─────────────────┼─────────────────────────────────┼────────────────────────────┤
│ product_subcategory            │          string │                       9 (90.0%) │                subcategory │
├────────────────────────────────┼─────────────────┼─────────────────────────────────┼────────────────────────────┤
│ target_age_range               │          string │                       5 (50.0%) │                   category │
├────────────────────────────────┼─────────────────┼─────────────────────────────────┼────────────────────────────┤
│ customer                       │            dict │                     10 (100.0%) │          person_from_faker │
├────────────────────────────────┼─────────────────┼─────────────────────────────────┼────────────────────────────┤
│ number_of_stars                │             int │                       3 (30.0%) │                    uniform │
├────────────────────────────────┼─────────────────┼─────────────────────────────────┼────────────────────────────┤
│ review_style                   │          string │                       4 (40.0%) │                   category │
└────────────────────────────────┴─────────────────┴─────────────────────────────────┴────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                📝 LLM-Text Columns                                                
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                       ┃               ┃                            ┃     prompt tokens ┃      completion tokens ┃
┃ column name           ┃     data type ┃       number unique values ┃        per record ┃             per record ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ product_name          │        string │                10 (100.0%) │      74.0 +/- 0.8 │            5.0 +/- 2.4 │
├───────────────────────┼───────────────┼─────────────────

## ⏭️ Next Steps

Now that you've seen the basics of Data Designer, check out the following notebooks to learn more about:

- [Structured outputs and jinja expressions](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/2-structured-outputs-and-jinja-expressions/)

- [Seeding synthetic data generation with an external dataset](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/3-seeding-with-a-dataset/)

- [Providing images as context](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/4-providing-images-as-context/)

- [Generating images](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/5-generating-images/)


## ⏭️ 次のステップ

Data Designerの基本を理解したところで、以下のノートブックでさらに詳しく学習しましょう。

- [構造化出力とJinja式](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/2-structured-outputs-and-jinja-expressions/)

- [外部データセットを使用した合成データ生成のシード](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/3-seeding-with-a-dataset/)

- [コンテキストとして画像を提供する](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/4-providing-images-as-context/)

- [画像を生成する](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/5-generating-images/)
